<a href="https://colab.research.google.com/github/ayushipradhan30/Repo-01/blob/main/Assignment_2_Solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2: Optimization Using Python
Only NumPy and Matplotlib are used. All optimizers are implemented from scratch (no optimization library performs the iterations).

In [ ]:
# ---------------------------------------------------------------
# Imports
# ---------------------------------------------------------------
import numpy as np                      # array maths, linear algebra
import matplotlib.pyplot as plt         # plotting
from matplotlib.patches import Arc      # used to draw the turn-angle markers in Q1(c)

%matplotlib inline

## Question 1: Exact line search for a quadratic objective
### (a) Implementation
For $J(w)=\tfrac12 w^TAw+b^Tw$ the gradient is $\nabla J(w)=Aw+b$. At each iteration:

$$g_t=-(Aw_t+b),\qquad \alpha_t=\frac{g_t^Tg_t}{g_t^TAg_t},\qquad w_{t+1}=w_t+\alpha_t g_t$$

The step $\alpha_t$ is the exact minimizer of $J(w_t+\alpha g_t)$ over $\alpha$. This follows from setting $\frac{d}{d\alpha}J(w_t+\alpha g_t)=0$.

In [ ]:
def exact_line_search_quadratic(A, b, w0, num_steps):
    """
    Steepest descent with exact line search on J(w) = 0.5 w^T A w + b^T w.

    Parameters
    ----------
    A         : (n, n) symmetric positive-definite matrix
    b         : (n,) vector
    w0        : (n,) initial point
    num_steps : maximum number of iterations (non-negative int)

    Returns
    -------
    iterates : list of copies of w_0, w_1, ..., (at most num_steps + 1 entries)
    """
    # Convert inputs to float arrays so integer inputs do not cause integer division / dtype issues
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)

    # .copy() so we never modify the caller's w0 and so stored iterates are independent arrays
    w = np.array(w0, dtype=float).copy()
    iterates = [w.copy()]                       # iterates[0] = w_0

    for _ in range(num_steps):
        grad = A @ w + b                        # gradient of J at the current point

        # Stopping criterion: gradient norm below 1e-10 means we are (numerically) at the minimizer
        if np.linalg.norm(grad) < 1e-10:
            break

        g = -grad                               # search direction = negative gradient
        Ag = A @ g                              # computed once, reused below
        denom = g @ Ag                          # g^T A g (strictly positive for SPD A and g != 0)

        # Safeguard against division by zero / invalid values
        # (cannot happen for SPD A with g != 0, but guards against bad input)
        if denom <= 0 or not np.isfinite(denom):
            break

        alpha = (g @ g) / denom                 # exact step size alpha_t = g^T g / g^T A g
        w = w + alpha * g                       # update: w_{t+1} = w_t + alpha_t g_t
        iterates.append(w.copy())               # store a copy of the new point

    return iterates

### (b) Data and numerical results
$A=\begin{bmatrix}2&0\\0&8\end{bmatrix}$, $b=[0,0]^T$, $w_0=[2,1]^T$, so $J(w_1,w_2)=w_1^2+4w_2^2$.

In [ ]:
# ---------------- Problem data ----------------
A  = np.array([[2., 0.],
               [0., 8.]])                       # SPD; eigenvalues 2 and 8 (condition number 4)
b  = np.array([0., 0.])
w0 = np.array([2.0, 1.0])

# Objective J(w) = 0.5 w^T A w + b^T w   (equals w1^2 + 4 w2^2 for this data)
J = lambda w: 0.5 * w @ A @ w + b @ w

# Run 10 steps of exact line search
its = exact_line_search_quadratic(A, b, w0, num_steps=10)

# ---- (b)(1) print w_0 and every returned iterate ----
print("1) Iterates (w_0 is the initial point)")
for t, w in enumerate(its):
    print(f"w_{t:<2d} = [{w[0]: .10e}, {w[1]: .10e}]")

# ---- (b)(2) print J(w_t) for every iterate ----
print("\n2) J(w_t)")
for t, w in enumerate(its):
    print(f"J(w_{t:<2d}) = {J(w):.10e}")

# ---- (b)(3) g_t^T g_{t+1} for t = 0..4 ----
# g_t is defined directly from the formula g_t = -(A w_t + b), independent of the optimizer's internals
print("\n3) g_t^T g_{t+1}, with g_t = -(A w_t + b)")
gs = [-(A @ w + b) for w in its]                # list of all negative gradients
for t in range(5):
    d = gs[t] @ gs[t + 1]                       # dot product of consecutive gradients
    # "Approximately zero" test: compare against a tolerance scaled by the size of g_t^T g_t,
    # since floating-point error is relative to the magnitude of the numbers involved
    is_zero = abs(d) < 1e-9 * max(1.0, gs[t] @ gs[t])
    print(f"g_{t}^T g_{t+1} = {d: .3e}   approx zero? {is_zero}")

**Observation:** every $g_t^Tg_{t+1}$ is zero to floating-point precision (about $10^{-15}$ or smaller). This is expected. The exact step makes $\frac{d}{d\alpha}J(w_t+\alpha g_t)=0$ at $w_{t+1}$, i.e. $\nabla J(w_{t+1})^Tg_t=0$, so consecutive gradients are orthogonal. This is why the path zigzags with ~90° turns.

### (c) Optimization path on the contour plot
Consecutive search directions are orthogonal, so the angle between the incoming segment $w_t-w_{t-1}$ and the outgoing segment $w_{t+1}-w_t$ should be about 90° at every intermediate iterate. A magenta arc marks this angle.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

# ---- Contour lines of J(w1, w2) = w1^2 + 4 w2^2 ----
x = np.linspace(-0.6, 2.4, 400)
y = np.linspace(-0.6, 1.3, 400)
X, Y = np.meshgrid(x, y)                        # 2-D grids of w1 and w2 values
Z = X**2 + 4 * Y**2                             # objective evaluated on the grid
cs = ax.contour(X, Y, Z, levels=[0.05, 0.2, 0.5, 1, 2, 3, 4, 6, 8], cmap="Blues")
ax.clabel(cs, fontsize=7)                       # print the J value on each contour

# ---- Path: line segments joining consecutive iterates, plus markers ----
P = np.array(its)                               # shape (num_iterates, 2)
ax.plot(P[:, 0], P[:, 1], "r-", lw=1.3, label="Path (segments)")
ax.plot(P[1:, 0], P[1:, 1], "ro", ms=4, label="Iterates $w_t$")        # w_1, w_2, ...
ax.plot(*P[0], "gs", ms=9, label="Initial point $w_0$")                # start point
ax.plot(0, 0, "k*", ms=14, label="Minimizer (0, 0)")                   # true minimizer

# Text labels for the first few points (later ones are too close together to label legibly)
for t, p in enumerate(P[:6]):
    ax.annotate(f"$w_{{{t}}}$", p, textcoords="offset points", xytext=(6, 6), fontsize=9)
ax.annotate("(0,0)", (0, 0), textcoords="offset points", xytext=(-30, -14), fontsize=9)

# ---- Turn-angle markers at every intermediate iterate w_1 ... w_{T-1} ----
first = True                                    # so the legend gets only one entry for the arcs
for t in range(1, len(P) - 1):
    inc = P[t - 1] - P[t]                       # vector from w_t back to w_{t-1} (reverse of incoming segment)
    out = P[t + 1] - P[t]                       # outgoing segment w_{t+1} - w_t

    # Arc radius: 30% of the shorter adjacent segment, so the arc always fits and shrinks with the path
    r = 0.3 * min(np.linalg.norm(inc), np.linalg.norm(out))

    # Angles (degrees) of the two rays leaving w_t; the Arc patch draws between them
    a1 = np.degrees(np.arctan2(inc[1], inc[0]))
    a2 = np.degrees(np.arctan2(out[1], out[0]))
    lo, hi = sorted([a1, a2])
    if hi - lo > 180:                           # choose the smaller (interior) angle, not the reflex one
        lo, hi = hi, lo + 360

    ax.add_patch(Arc(P[t], 2 * r, 2 * r, theta1=lo, theta2=hi, color="m", lw=1.6,
                     label=r"Turn angle $\approx 90^\circ$" if first else None))
    first = False

    # Write "90°" next to the arc for the first few (larger, readable) turns
    if t <= 4:
        mid = np.radians((lo + hi) / 2)         # direction bisecting the arc
        ax.text(P[t, 0] + 1.9 * r * np.cos(mid), P[t, 1] + 1.9 * r * np.sin(mid),
                "90°", color="m", fontsize=8)

# ---- Cosmetics required by the assignment ----
ax.set_xlabel("$w_1$")
ax.set_ylabel("$w_2$")
ax.set_title(r"Exact line search on $J=w_1^2+4w_2^2$: path $w_0\to\cdots\to w_{10}$")
ax.set_aspect("equal")                          # equal scaling so 90 degree angles look like 90 degrees
ax.grid(True)
ax.legend(loc="upper right")
plt.show()

# ---- Numerical check: measure the actual angle between incoming and outgoing segments ----
print("Angle between incoming and outgoing segments at each intermediate iterate:")
for t in range(1, len(P) - 1):
    u, v = P[t] - P[t - 1], P[t + 1] - P[t]
    cosang = u @ v / (np.linalg.norm(u) * np.linalg.norm(v))   # cos(angle) = u.v / (|u||v|)
    print(f"  at w_{t}: {np.degrees(np.arccos(cosang)):.4f} deg")

## Question 2: Gradient descent, Momentum and RMSProp
Loss: $J(w_1,w_2)=w_1^2+10w_2^2$, with gradient $\nabla J=[2w_1,\;20w_2]^T$. The Hessian eigenvalues are 2 and 20, so the problem is ill-conditioned (ratio 10).

### (a) Optimizers

In [ ]:
def vanilla_gd(grad_fn, w0, alpha, num_iters):
    """Plain gradient descent: w <- w - alpha * grad(w)."""
    w = np.array(w0, dtype=float)               # float copy of the start point
    its = [w.copy()]                            # its[0] = w_0
    for _ in range(num_iters):
        w = w - alpha * grad_fn(w)              # step against the gradient
        its.append(w.copy())                    # record every iterate
    return its


def momentum_gd(grad_fn, w0, alpha, beta, num_iters):
    """
    Gradient descent with momentum (velocity accumulates past gradients):
        v_{t+1} = beta * v_t + grad(w_t),   w_{t+1} = w_t - alpha * v_{t+1},   v_0 = 0
    beta in [0, 1) controls how much past velocity is kept.
    """
    w = np.array(w0, dtype=float)
    v = np.zeros_like(w)                        # velocity starts at zero
    its = [w.copy()]
    for _ in range(num_iters):
        v = beta * v + grad_fn(w)               # update velocity using gradient at the OLD point w_t
        w = w - alpha * v                       # move along the velocity
        its.append(w.copy())
    return its


def rmsprop_gd(grad_fn, w0, alpha, rho, eps, num_iters):
    """
    RMSProp: scale each coordinate's step by a running RMS of its gradient.
        s_{t+1} = rho * s_t + (1 - rho) * g_t (.) g_t,   s_0 = 0
        w_{t+1} = w_t - alpha * g_t / (sqrt(s_{t+1}) + eps)
    All operations are element-wise; eps avoids division by zero.
    """
    w = np.array(w0, dtype=float)
    s = np.zeros_like(w)                        # running average of squared gradients
    its = [w.copy()]
    for _ in range(num_iters):
        g = grad_fn(w)                          # evaluate the gradient ONCE per iteration and reuse it
        s = rho * s + (1 - rho) * (g * g)       # g*g is element-wise square (the Hadamard product)
        w = w - alpha * g / (np.sqrt(s) + eps)  # per-coordinate normalised step
        its.append(w.copy())
    return its

### (b)(1) Run all three methods
Common settings: $w_0=[3,1]^T$, $\alpha=0.05$, 50 iterations. Chosen hyperparameters: **$\beta=0.9$** (standard momentum value), **$\rho=0.9$** (standard RMSProp decay), **$\varepsilon=10^{-8}$** (small constant avoiding division by zero).

In [ ]:
# ---- Loss and its gradient ----
J2    = lambda w: w[0]**2 + 10 * w[1]**2        # J(w1, w2) = w1^2 + 10 w2^2
grad2 = lambda w: np.array([2 * w[0], 20 * w[1]])   # [dJ/dw1, dJ/dw2]

# ---- Shared settings (identical for all three methods, as required) ----
w0    = np.array([3.0, 1.0])
alpha = 0.05
N     = 50

# ---- Method-specific hyperparameters (chosen and stated here) ----
beta = 0.9      # momentum coefficient
rho  = 0.9      # RMSProp decay rate of the squared-gradient average
eps  = 1e-8     # RMSProp numerical-stability constant

# Run each optimizer; each returns a list of N+1 points (w_0 plus one point per update)
runs = {
    "Vanilla GD":                    vanilla_gd(grad2, w0, alpha, N),
    f"Momentum (β={beta})":          momentum_gd(grad2, w0, alpha, beta, N),
    f"RMSProp (ρ={rho}, ε={eps:g})": rmsprop_gd(grad2, w0, alpha, rho, eps, N),
}

# Summary of where each method ended up after 50 iterations
for name, path in runs.items():
    print(f"{name:28s} final w = {path[-1]},  J = {J2(path[-1]):.3e}")

### (b)(2) Loss vs iteration (log scale)
If a loss becomes exactly 0 it cannot be drawn on a log axis, so losses are floored at $10^{-16}$ for display only. The underlying values are not changed.

In [ ]:
FLOOR = 1e-16                                   # display floor so log(0) is never evaluated

fig, ax = plt.subplots(figsize=(8, 5))
for name, path in runs.items():
    losses = [J2(w) for w in path]              # J(w_t) for t = 0..N
    losses = np.maximum(losses, FLOOR)          # replace any value below FLOOR (including 0) by FLOOR
    ax.semilogy(range(len(path)), losses, lw=1.8, label=name)   # log-scaled y-axis

ax.set_xlabel("Iteration $t$")
ax.set_ylabel(r"$J(w_t)$ (log scale; floored at $10^{-16}$)")
ax.set_title("Loss vs iteration: $J=w_1^2+10w_2^2$")
ax.grid(True, which="both", alpha=0.4)          # "both" shows major and minor grid lines on the log axis
ax.legend()
plt.show()

### (b)(3) Optimization paths over contours
All three panels share the same contour levels and the same axis limits, so the paths are directly comparable.

In [ ]:
# ---- Common axis limits: computed from ALL paths so every panel shows every trajectory fully ----
allp = np.vstack([np.array(path) for path in runs.values()])
xl = (allp[:, 0].min() - 0.3, allp[:, 0].max() + 0.3)
yl = (allp[:, 1].min() - 0.3, allp[:, 1].max() + 0.3)

# ---- Common contour grid and levels ----
X, Y = np.meshgrid(np.linspace(*xl, 400), np.linspace(*yl, 400))
Z = X**2 + 10 * Y**2
levels = [0.1, 0.5, 1, 2, 4, 8, 12, 16, 20]     # identical in every panel

# sharex/sharey keep the axes identical across the three subplots
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)

for ax, (name, path) in zip(axes, runs.items()):
    P = np.array(path)
    ax.contour(X, Y, Z, levels=levels, cmap="Blues")                 # level sets of J
    ax.plot(P[:, 0], P[:, 1], "r.-", lw=1, ms=4, label="Path")       # optimizer trajectory
    ax.plot(*P[0], "gs", ms=8, label="$w_0$")                        # starting point
    ax.plot(0, 0, "k*", ms=12, label="Minimizer")                    # true minimum at the origin
    ax.set_xlim(xl); ax.set_ylim(yl)                                 # same limits in every panel
    ax.set_title(name)
    ax.set_xlabel("$w_1$")
    ax.grid(True, alpha=0.4)
    ax.legend()

axes[0].set_ylabel("$w_2$")                      # y-label only on the left panel (axes are shared)
fig.suptitle("Optimization paths over contours of $J=w_1^2+10w_2^2$ (same levels & limits)")
plt.show()

### Discussion
- **Vanilla GD:** the curvature along $w_2$ is 20 and $\alpha=0.05$, so $\alpha\cdot 20=1$. The $w_2$ coordinate goes to 0 in a single step, and $w_1$ then shrinks by a factor $1-2\alpha=0.9$ per step. It has the lowest final loss here.
- **Momentum ($\beta=0.9$):** the accumulated velocity overshoots the valley floor, so the path oscillates in the steep $w_2$ direction. It is still ringing down at iteration 50.
- **RMSProp:** dividing by $\sqrt{s}$ makes every coordinate take steps of size about $\alpha$ regardless of gradient size. $w_2$ settles quickly, but $w_1$ moves only about 0.05 per step, so it is still far from 0 after 50 iterations. A smaller or decaying $\alpha$ would help.

These conclusions depend on the chosen $\beta,\rho,\varepsilon$.